# X Gate and CNOT Control-Target Demonstration

---

## 1. Problem Statement
This notebook demonstrates how a **Pauli-X gate** prepares the control qubit in the state $|1\rangle$, followed by a **Controlled-NOT (CNOT / CX) gate** that conditionally flips the target qubit from $|0\rangle$ to $|1\rangle$ when the control qubit is in state $|1\rangle$.

The objective is to establish and verify deterministic conditional state transitions in a two-qubit quantum circuit using Qiskit Aer simulation and automated result validation.

---

## 2. Theoretical Concepts & Component Roles

### Key Quantum Concepts & Gates
- **Pauli-X Gate ($X$)**: A single-qubit quantum gate performing a bit-flip operation, mapping $|0\rangle \to |1\rangle$ and $|1\rangle \to |0\rangle$. Mathematically represented by the Pauli matrix:
  $$X = \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix}$$
  In this circuit, the X gate is applied to qubit 0 to prepare it into the excited computational basis state $|1\rangle$.
- **CNOT / CX Gate**: A two-qubit controlled unitary gate that applies a Pauli-X (NOT) operation to the target qubit if and only if the control qubit is in state $|1\rangle$; otherwise, the target qubit remains unchanged. Its computational action is:
  $$\text{CNOT}|c, t\rangle = |c, c \oplus t\rangle$$
  where $\oplus$ denotes modulo-2 addition (XOR). In standard computational matrix form:
  $$\text{CNOT} = \begin{pmatrix} 1 & 0 & 0 & 0 \\ 0 & 1 & 0 & 0 \\ 0 & 0 & 0 & 1 \\ 0 & 0 & 1 & 0 \end{pmatrix}$$
- **Control Qubit ($q_0$)**: The conditioning qubit that determines whether the operation on the target qubit is executed. Its own computational basis state remains unchanged by the CNOT operation.
- **Target Qubit ($q_1$)**: The controlled qubit whose state is conditionally inverted based on the state of the control qubit.
- **Measurement**: Projective measurement in the computational $Z$-basis ($\{|0\rangle, |1\rangle\}$). It collapses the quantum state of each qubit into a definitive classical bit ($0$ or $1$) stored in a designated classical register bit ($c_0$ and $c_1$). Note that Qiskit represents classical bitstrings in little-endian order ($c_1 c_0$).
- **Shot-Based Simulation**: Executing the circuit across multiple independent statistical iterations ("shots", here 1024) via Qiskit Aer's `AerSimulator` to collect empirical measurement counts and verify deterministic behavior.

> **Technical Note on State Separability & Entanglement**:  
> While the CNOT gate is widely recognized as an entangling gate when preceded by a superposition state (e.g., $H|0\rangle$), this circuit **does not demonstrate entanglement**. The state transition proceeds deterministically strictly within computational basis states:
> $$|00\rangle \xrightarrow{X_0} |01\rangle \xrightarrow{\text{CX}_{0,1}} |11\rangle = |1\rangle \otimes |1\rangle$$
> Because no superposition is created, the final state is the separable product state $|11\rangle = |1\rangle \otimes |1\rangle$, and no quantum entanglement is generated.

---

## 3. Circuit Walkthrough & Step-by-Step Logic

The quantum circuit execution proceeds through the following specific stages:

1. **Register Allocation**: The circuit contains 2 qubits ($q_0, q_1$) and 2 classical bits ($c_0, c_1$), initialized in the ground state $|q_1 q_0\rangle = |00\rangle$.
2. **Control State Preparation (`qc.x(0)`)**: `qc.x(0)` changes qubit 0 from $|0\rangle$ to $|1\rangle$.
3. **Controlled Gate Application (`qc.cx(0, 1)`)**: `qc.cx(0, 1)` uses qubit 0 as the control and qubit 1 as the target.
4. **Conditional Target Inversion**: Because the control qubit is $|1\rangle$, the target qubit flips from $|0\rangle$ to $|1\rangle$.
5. **State Projection**: Both qubits are now in state $|1\rangle$; therefore the expected measured computational-basis state is $|11\rangle$.
6. **Execution via AerSimulator**: The circuit is executed with `AerSimulator` for **1024 shots** to sample the measurement distribution.
7. **Automated Result Check**: The existing result check verifies whether all 1024 shots produce `'11'`, programmatically confirming correct circuit behavior.

---

## 4. Environment Setup
Install necessary dependencies (`qiskit` and `qiskit-aer`).

In [1]:
pip install qiskit

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


---

## 5. Quantum Circuit Construction & Simulation
Construct the 2-qubit, 2-classical-bit circuit, apply the Pauli-X and CNOT gates, perform measurement into classical registers, simulate using `AerSimulator` for 1024 shots, and verify the resulting counts.

In [2]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

qc = QuantumCircuit(2, 2)

qc.x(0)
qc.cx(0, 1)

qc.measure(0, 0)
qc.measure(1, 1)

print(qc.draw())

simulator = AerSimulator()

job = simulator.run(qc, shots=1024)
result = job.result()
counts = result.get_counts()

print(counts)

if counts == {'11': 1024}:
    print("SUCCESS!")
    print("Control qubit = |1>")
    print("Target qubit flipped from |0> to |1>.")
else:
    print("Unexpected result:", counts)

/var/folders/yp/078pyxw11mq6g7sd_2ty18z80000gn/T/ipykernel_2010/2551537933.py:1: DeprecationWarning: Using Qiskit with Python 3.9 is deprecated as of the 2.1.0 release. Support for running Qiskit with Python 3.9 will be removed in the 2.3.0 release, which coincides with when Python 3.9 goes end of life.
  from qiskit import QuantumCircuit


     ┌───┐     ┌─┐   
q_0: ┤ X ├──■──┤M├───
     └───┘┌─┴─┐└╥┘┌─┐
q_1: ─────┤ X ├─╫─┤M├
          └───┘ ║ └╥┘
c: 2/═══════════╩══╩═
                0  1 
{'11': 1024}
SUCCESS!
Control qubit = |1>
Target qubit flipped from |0> to |1>.


---

## 6. Expected Results & Conclusion

### Expected Results
- **Theoretical State**: $|11\rangle$
- **Expected Ideal Simulation Result**: `{'11': 1024}` (all 1024 shots result in outcome `'11'`)
- **Verification Output**:
  ```text
  SUCCESS!
  Control qubit = |1>
  Target qubit flipped from |0> to |1>.
  ```

### Conclusion
This demonstration verifies the fundamental mechanism of conditional bit flipping using a CNOT gate:
- The Pauli-X gate prepares the control qubit in state $|1\rangle$.
- The CNOT gate detects the control qubit in state $|1\rangle$ and deterministically flips the target qubit from $|0\rangle$ to $|1\rangle$.
- The simulation produces the exact expected count of `{'11': 1024}`.
- Because no superposition state was introduced, the final state is a separable product state ($|1\rangle \otimes |1\rangle$) and does not exhibit quantum entanglement, serving as a clean demonstration of reversible quantum logic.